In [2]:
import pandas as pd
from pathlib import Path
import numpy as np
import zipfile
import pickle
import io
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

In [3]:
Project_root = Path.cwd().parent
mission_1 = Project_root / 'Data' /'ESA-Mission1'/ 'ESA-Mission1'

channels = mission_1 / 'channels.csv'
labels = mission_1 / 'labels.csv'
anaomaly_types = mission_1 / 'anomaly_types.csv'
telemetry_dir = mission_1 / 'channels'

channels_df = pd.read_csv(channels)
labels_df = pd.read_csv(labels)
anomaly_types_df = pd.read_csv(anaomaly_types)

print(channels_df.shape)
print(labels_df.shape)
print(anomaly_types_df.shape)

(76, 5)
(3589, 4)
(200, 7)


In [4]:
# 1. Check channel information
print("Channel shape:", channels_df.shape)
print(channels_df.info())

print("\nTarget distribution:")
print(channels_df["Target"].value_counts())

# 2. Check annotation information
print("\nLabels shape:", labels_df.shape)
print(labels_df.info())

# 3. Check anomaly categories
print("\nAnomaly categories:")
print(anomaly_types_df["Category"].value_counts())

# 4. Check relationship between labels and anomaly types
annotation_df = labels_df.merge(
    anomaly_types_df,
    on="ID",
    how="left"
)

print("\nCombined annotations:")
print(annotation_df.head())

Channel shape: (76, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Channel        76 non-null     object
 1   Subsystem      76 non-null     object
 2   Physical Unit  76 non-null     object
 3   Group          76 non-null     int64 
 4   Target         76 non-null     object
dtypes: int64(1), object(4)
memory usage: 3.1+ KB
None

Target distribution:
Target
YES    58
NO     18
Name: count, dtype: int64

Labels shape: (3589, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3589 entries, 0 to 3588
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         3589 non-null   object
 1   Channel    3589 non-null   object
 2   StartTime  3589 non-null   object
 3   EndTime    3589 non-null   object
dtypes: object(4)
memory usage: 112.3+ KB
None

Anomaly categories:
Category
An

In [5]:
telemetry_files = sorted(telemetry_dir.iterdir())
print("total telemetry files:", len(telemetry_files))

for file in telemetry_files[:10]:
    print(file.name, file.suffix, file.stat().st_size)

total telemetry files: 76
channel_1  0
channel_10  0
channel_11  0
channel_12  0
channel_13  0
channel_14  0
channel_15  0
channel_16  0
channel_17  0
channel_18  0


In [6]:
processed_dir = Project_root/"data"/"processed"/"ESA-mission1"
processed_dir.mkdir(parents = True, exist_ok= True)

print(Project_root)
print(mission_1)
print(mission_1.exists())

c:\Users\venky\Desktop\Hochschule Hof\Hof Sem 2\Data Mining And Machine Learning
c:\Users\venky\Desktop\Hochschule Hof\Hof Sem 2\Data Mining And Machine Learning\Data\ESA-Mission1\ESA-Mission1
True


In [7]:
target_channels = channels_df.loc[
    channels_df["Target"].str.upper() == "YES",
    "Channel"
].tolist()

print("Number of target channel :", len(target_channels))
print(target_channels)

Number of target channel : 58
['channel_12', 'channel_13', 'channel_14', 'channel_15', 'channel_16', 'channel_17', 'channel_18', 'channel_19', 'channel_20', 'channel_21', 'channel_22', 'channel_23', 'channel_24', 'channel_25', 'channel_26', 'channel_27', 'channel_28', 'channel_29', 'channel_30', 'channel_31', 'channel_32', 'channel_33', 'channel_34', 'channel_35', 'channel_36', 'channel_37', 'channel_38', 'channel_39', 'channel_40', 'channel_41', 'channel_42', 'channel_43', 'channel_44', 'channel_45', 'channel_46', 'channel_47', 'channel_48', 'channel_49', 'channel_50', 'channel_51', 'channel_52', 'channel_57', 'channel_58', 'channel_59', 'channel_60', 'channel_61', 'channel_62', 'channel_63', 'channel_64', 'channel_65', 'channel_66', 'channel_70', 'channel_71', 'channel_72', 'channel_73', 'channel_74', 'channel_75', 'channel_76']


In [8]:
annotation_df = labels_df.merge(
    anomaly_types_df,
    on ="ID",
    how = 'left',
    validate= "many_to_one"
)

annotation_df["StartTime"] = pd.to_datetime(
    annotation_df["StartTime"], utc=True
)

annotation_df['EndTime'] = pd.to_datetime(
    annotation_df['EndTime'], utc = True
)

target_annotations = annotation_df[
    annotation_df["Channel"].isin(target_channels)
].copy()

print(target_annotations["Category"].value_counts())

Category
Rare Event           2178
Anomaly              1203
Communication Gap     208
Name: count, dtype: int64


In [9]:
telemetry_files = sorted(telemetry_dir.iterdir())
print("total entries: ", len(telemetry_files))

for file in telemetry_files[:15]:
    print(
        file.name,
        "| file:", file.is_file(),
        "| directory:", file.is_dir(),
        "| size:", file.stat().st_size
        )

total entries:  76
channel_1 | file: False | directory: True | size: 0
channel_10 | file: False | directory: True | size: 0
channel_11 | file: False | directory: True | size: 0
channel_12 | file: False | directory: True | size: 0
channel_13 | file: False | directory: True | size: 0
channel_14 | file: False | directory: True | size: 0
channel_15 | file: False | directory: True | size: 0
channel_16 | file: False | directory: True | size: 0
channel_17 | file: False | directory: True | size: 0
channel_18 | file: False | directory: True | size: 0
channel_19 | file: False | directory: True | size: 0
channel_2 | file: False | directory: True | size: 0
channel_20 | file: False | directory: True | size: 0
channel_21 | file: False | directory: True | size: 0
channel_22 | file: False | directory: True | size: 0


In [10]:
sample_file = telemetry_dir/ "channel_2"/ "channel_2"

if sample_file.is_dir():
    for item in sample_file.rglob("*"):
        print(item, item.stat().st_size)
print("exist:", sample_file.exists())
print("is_file: ", sample_file.is_file())
print("is_directory", sample_file.is_dir())

with open(sample_file, "rb") as f:
    file_signature = f.read(20)
print(file_signature) #we found its a pickle file
print("file_size", sample_file.stat().st_size)

exist: True
is_file:  True
is_directory False
b'\x80\x05\x95\xb4\x00\x00\x00\x00\x00\x00\x00\x8c\x11pandas.'
file_size 126161031


In [11]:
#loading our first telemetry data

sample_channel = "channel_2"
sample_path = telemetry_dir/ sample_channel/ sample_channel
sample_df = pd.read_pickle(sample_path)

print(sample_df.head())
print("colums", sample_df.columns)
print("data types : ", sample_df.dtypes)
print(sample_df.shape)
print(sample_df.index)

                         channel_2
datetime                          
2000-01-01 00:00:02.805   0.126334
2000-01-01 00:01:32.805   0.126334
2000-01-01 00:03:02.805   0.126334
2000-01-01 00:04:32.805   0.126334
2000-01-01 00:06:02.805   0.126334
colums Index(['channel_2'], dtype='object')
data types :  channel_2    float32
dtype: object
(10513336, 1)
DatetimeIndex(['2000-01-01 00:00:02.805000', '2000-01-01 00:01:32.805000',
               '2000-01-01 00:03:02.805000', '2000-01-01 00:04:32.805000',
               '2000-01-01 00:06:02.805000', '2000-01-01 00:07:32.805000',
               '2000-01-01 00:09:02.805000', '2000-01-01 00:10:32.805000',
               '2000-01-01 00:12:02.805000', '2000-01-01 00:13:32.805000',
               ...
               '2013-12-31 23:46:26.109000', '2013-12-31 23:47:56.109000',
               '2013-12-31 23:49:26.109000', '2013-12-31 23:50:56.109000',
               '2013-12-31 23:52:26.109000', '2013-12-31 23:53:56.109000',
               '2013-12-31 23

In [12]:
#our function to call channels
def load_channel(channel):
    file_path = telemetry_dir/ channel / channel
    return pd.read_pickle(file_path)

In [13]:
telemetry_data = {}
for channel in target_channels:
    telemetry_data[channel] = load_channel(channel)

print("loaded_channels: ", len(telemetry_data))

loaded_channels:  58


In [14]:
print(telemetry_data.keys())

dict_keys(['channel_12', 'channel_13', 'channel_14', 'channel_15', 'channel_16', 'channel_17', 'channel_18', 'channel_19', 'channel_20', 'channel_21', 'channel_22', 'channel_23', 'channel_24', 'channel_25', 'channel_26', 'channel_27', 'channel_28', 'channel_29', 'channel_30', 'channel_31', 'channel_32', 'channel_33', 'channel_34', 'channel_35', 'channel_36', 'channel_37', 'channel_38', 'channel_39', 'channel_40', 'channel_41', 'channel_42', 'channel_43', 'channel_44', 'channel_45', 'channel_46', 'channel_47', 'channel_48', 'channel_49', 'channel_50', 'channel_51', 'channel_52', 'channel_57', 'channel_58', 'channel_59', 'channel_60', 'channel_61', 'channel_62', 'channel_63', 'channel_64', 'channel_65', 'channel_66', 'channel_70', 'channel_71', 'channel_72', 'channel_73', 'channel_74', 'channel_75', 'channel_76'])


In [15]:
print(telemetry_data["channel_12"].head())
print(telemetry_data["channel_13"].shape)

                         channel_12
datetime                           
2000-01-01 00:00:06.633    0.317175
2000-01-01 00:01:36.633    0.317175
2000-01-01 00:03:06.633    0.317175
2000-01-01 00:04:36.633    0.317175
2000-01-01 00:06:06.633    0.317175
(14258506, 1)


In [16]:
total_memory = sum(
    df.memory_usage(deep=True).sum()
    for df in telemetry_data.values()
)
print(f"Total_memory: {total_memory / 1024**3:.2f} GB")

#we now have all 58 telemetry data loaded here

Total_memory: 6.84 GB


In [33]:
def inspect_channels(telemetry_data):
    for channel, df in telemetry_data.items():
        print(
            channel,
            "| shape:", df.shape,
            "| start:", df.index.min(),
            "| end:", df.index.max(),
            "| memory", round(
                df.memory_usage(deep=True).sum()/(1024**2), 2
            ), "MB"
        )
inspect_channels(telemetry_data)


channel_12 | shape: (14258506, 1) | start: 2000-01-01 00:00:06.633000 | end: 2013-12-31 23:59:59.937000 | memory 163.18 MB
channel_13 | shape: (14258506, 1) | start: 2000-01-01 00:00:06.633000 | end: 2013-12-31 23:59:59.937000 | memory 163.18 MB
channel_14 | shape: (5272893, 1) | start: 2000-01-01 00:00:06.633000 | end: 2013-12-31 23:59:59.937000 | memory 60.34 MB
channel_15 | shape: (5272893, 1) | start: 2000-01-01 00:00:06.633000 | end: 2013-12-31 23:59:59.937000 | memory 60.34 MB
channel_16 | shape: (5272893, 1) | start: 2000-01-01 00:00:06.633000 | end: 2013-12-31 23:59:59.937000 | memory 60.34 MB
channel_17 | shape: (14258506, 1) | start: 2000-01-01 00:00:06.633000 | end: 2013-12-31 23:59:59.937000 | memory 163.18 MB
channel_18 | shape: (14258506, 1) | start: 2000-01-01 00:00:06.633000 | end: 2013-12-31 23:59:59.937000 | memory 163.18 MB
channel_19 | shape: (14258506, 1) | start: 2000-01-01 00:00:06.633000 | end: 2013-12-31 23:59:59.937000 | memory 163.18 MB
channel_20 | shape: (1

In [36]:
def check_quality(telemetry_data):
    for channel, df in telemetry_data.items():
        print(
            channel,
            "| missing", df.isna().sum().sum(),
            "| duplicate ", df.index.duplicated().sum(),
            "| interval", df.index.diff().value_counts()
        )
check_quality(telemetry_data)

channel_12 | missing 0 | duplicate  0 | interval datetime
0 days 00:00:00.375000    7653394
0 days 00:01:30           4812347
0 days 00:00:00.378000     613497
0 days 00:00:00.372000     555289
0 days 00:00:03            356283
                           ...   
0 days 05:12:00                 1
0 days 00:00:10.875000          1
0 days 00:01:26.625000          1
0 days 00:00:15.798000          1
0 days 00:01:14.202000          1
Name: count, Length: 134, dtype: int64
channel_13 | missing 0 | duplicate  0 | interval datetime
0 days 00:00:00.375000    7653394
0 days 00:01:30           4812347
0 days 00:00:00.378000     613497
0 days 00:00:00.372000     555289
0 days 00:00:03            356283
                           ...   
0 days 05:12:00                 1
0 days 00:00:10.875000          1
0 days 00:01:26.625000          1
0 days 00:00:15.798000          1
0 days 00:01:14.202000          1
Name: count, Length: 134, dtype: int64
channel_14 | missing 0 | duplicate  0 | interval datetime


In [42]:
def sampling_summary(telemetry_data):
    results = []
    for channel, df in telemetry_data.items():
        intervals = df.index.to_series().diff().dropna()
        most_common = intervals.mode().iloc[0]

        results.append({
            "channel" : channel,
            "Most common interval": most_common,
            "unique intervals ": intervals.nunique(),
            "largest_gap" : intervals.max()
        })
    return pd.DataFrame(results)

sampling_df = sampling_summary(telemetry_data)
print(sampling_df.to_string(index=False))

   channel   Most common interval  unique intervals              largest_gap
channel_12 0 days 00:00:00.375000                134         0 days 05:55:30
channel_13 0 days 00:00:00.375000                134         0 days 05:55:30
channel_14        0 days 00:01:30                132         0 days 05:55:30
channel_15        0 days 00:01:30                132         0 days 05:55:30
channel_16        0 days 00:01:30                132         0 days 05:55:30
channel_17 0 days 00:00:00.375000                134         0 days 05:55:30
channel_18 0 days 00:00:00.375000                134         0 days 05:55:30
channel_19 0 days 00:00:00.375000                134         0 days 05:55:30
channel_20 0 days 00:00:00.375000                134         0 days 05:55:30
channel_21        0 days 00:01:30                132         0 days 05:55:30
channel_22        0 days 00:01:30                132         0 days 05:55:30
channel_23        0 days 00:01:30                132         0 days 05:55:30

In [45]:
start_times = [
    df.index.min()
    for df in telemetry_data.values()
]

end_times = [
    df.index.max()
    for df in telemetry_data.values()
]

common_start = max(start_times)
common_end = min(end_times)

print("latest channel Start :", common_start)
print("Earliest channel end", common_end)

common_timeline =pd.date_range(
    start = common_start,
    end = common_end,
    freq = "30s"
)
print("Total timestamps", len(common_timeline))
print("First :", common_timeline[0])
print("End :", common_timeline[-1])

latest channel Start : 2000-01-01 00:06:45.177000
Earliest channel end 2013-12-31 23:57:38.481000
Total timestamps 14728302
First : 2000-01-01 00:06:45.177000
End : 2013-12-31 23:57:15.177000


In [48]:
#sample for aligning a channel
test_channel = telemetry_data["channel_12"]
aligned_test = test_channel.reindex(
    common_timeline,
    method ="ffill"
)
print(aligned_test.shape)
print(aligned_test.isna().sum())
print(aligned_test.head())

(14728302, 1)
channel_12    0
dtype: int64
                         channel_12
2000-01-01 00:06:45.177    0.317175
2000-01-01 00:07:15.177    0.317175
2000-01-01 00:07:45.177    0.317175
2000-01-01 00:08:15.177    0.317175
2000-01-01 00:08:45.177    0.317175


In [50]:
aligned_channels = {}

for channel, df in telemetry_data.items():
    aligned_channels[channel] =df.reindex(
        common_timeline,
        method = "ffill"
    )[channel]

telemetry_df = pd.DataFrame(aligned_channels)
print("shape:", telemetry_df.shape)
print("missing values :", telemetry_df.isna().sum().sum())
print(telemetry_df.info())

shape: (14728302, 58)
missing values : 0
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 14728302 entries, 2000-01-01 00:06:45.177000 to 2013-12-31 23:57:15.177000
Freq: 30s
Data columns (total 58 columns):
 #   Column      Dtype  
---  ------      -----  
 0   channel_12  float32
 1   channel_13  float32
 2   channel_14  float32
 3   channel_15  float32
 4   channel_16  float32
 5   channel_17  float32
 6   channel_18  float32
 7   channel_19  float32
 8   channel_20  float32
 9   channel_21  float32
 10  channel_22  float32
 11  channel_23  float32
 12  channel_24  float32
 13  channel_25  float32
 14  channel_26  float32
 15  channel_27  float32
 16  channel_28  float32
 17  channel_29  float32
 18  channel_30  float32
 19  channel_31  float32
 20  channel_32  float32
 21  channel_33  float32
 22  channel_34  float32
 23  channel_35  float32
 24  channel_36  float32
 25  channel_37  float32
 26  channel_38  float32
 27  channel_39  float32
 28  channel_40  float32
 29  channel_

In [52]:
telemetry_df.to_parquet(
    "mission1_aligned.parquet",
    engine ="pyarrow",
    compression="snappy",
    index = True
)

In [63]:
# removing telemetry df from memory
import gc

# del telemetry_df
# del aligned_channels
del check_df
gc.collect()

1537

In [17]:
import os
file_path = processed_dir/ "mission1_aligned.parquet"
print("file exists", file_path.exists())

print("file size:", os.path.getsize(file_path) / (1024**3), "GB")
check_df = pd.read_parquet(file_path)
print("shape:", check_df.shape)
print(check_df.dtypes.value_counts())

file exists True
file size: 0.38955019786953926 GB
shape: (14728302, 58)
float32    58
Name: count, dtype: int64
